# Agent Design · Day 29 — **Agent Testing & Quality Gates**

**Phase 1 · Module 4: Agentic System Design Patterns** · ~2.5 hrs

Agents are non-deterministic, call external tools, and change behaviour when a prompt is tweaked. You cannot ship them on hope. Four testing layers make them safe to change: **unit tests** per node, **contract tests** at tool boundaries (the PACT idea), **mutation tests** that check your tests actually bite, and an automated **quality gate** that fails the build when eval scores drop.

Today:
1. The agent under test — plain node functions.
2. **Unit tests** per node (fast, deterministic).
3. **Contract test** — a consumer's expectation the provider must honour.
4. **Mutation testing** — break the code on purpose; do the tests catch it?
5. A **quality gate** — threshold on an eval suite, pass/fail.

All keyless and runnable inline with `assert`.

## 1. The agent under test

Three node functions from a loan-eligibility agent: `validate` the input, `score` the applicant, `decide` the outcome. Small, pure, deterministic — which is exactly what makes them testable.

In [1]:
def validate(app):
    if not isinstance(app.get("income"), (int, float)) or app["income"] < 0:
        raise ValueError("income must be a non-negative number")
    if not (0 <= app.get("credit", -1) <= 999):
        raise ValueError("credit score out of range")
    return app

def score(app):
    # simple risk score: higher is better
    return app["credit"] / 999 * 0.6 + min(app["income"] / 100_000, 1) * 0.4

def decide(app, threshold=0.5):
    return {"approved": score(app) >= threshold, "score": round(score(app), 3)}

demo = validate({"income": 60_000, "credit": 720})
print(decide(demo))

{'approved': True, 'score': 0.672}


☝ Each node does one thing and returns a value — no hidden state, no I/O. That design *is* the testability: you can call `decide` a thousand times with crafted inputs and assert on the output. If your nodes reach into a database or an LLM directly, you can't; mock those out (today's Python lesson) so the logic stays unit-testable.

## 2. Unit tests per node

One test per behaviour: valid input passes, invalid input raises, the score is monotonic, the boundary decision is correct. Run them and count.

In [2]:
def run_tests(tests):
    passed = 0
    for name, fn in tests:
        try:
            fn(); print(f"PASS {name}"); passed += 1
        except AssertionError as e:
            print(f"FAIL {name}: {e}")
    print(f"--- {passed}/{len(tests)} passed")

def t_valid():   assert validate({"income": 10, "credit": 500})["credit"] == 500
def t_bad_income():
    try: validate({"income": -1, "credit": 500}); assert False, "should raise"
    except ValueError: pass
def t_monotonic(): assert score({"income": 50_000, "credit": 800}) > score({"income": 50_000, "credit": 400})
def t_boundary(): assert decide({"income": 100_000, "credit": 999})["approved"] is True

run_tests([("valid", t_valid), ("bad_income", t_bad_income),
           ("monotonic", t_monotonic), ("boundary", t_boundary)])

PASS valid
PASS bad_income
PASS monotonic
PASS boundary
--- 4/4 passed


☝ These run in microseconds and pin down the *deterministic* core of the agent — the parts that must never regress silently. In a real repo these are `pytest` functions (Day 24) run on every commit. The rule: every node function gets at least a happy-path test and a failure-path test before it's considered done.

## 3. Contract test — honour the consumer's expectation

A **contract** (the PACT pattern) is a shared promise between a *consumer* (whatever calls the agent) and the *provider* (the agent). The consumer declares the shape it needs; the test fails if the provider ever stops delivering it — catching breaking changes before they reach the caller.

In [3]:
# The consumer (e.g. the loan UI) depends on exactly this shape:
CONTRACT = {
    "approved": bool,
    "score": float,
}

def verify_contract(output, contract):
    for key, typ in contract.items():
        assert key in output, f"missing field '{key}'"
        assert isinstance(output[key], typ), f"'{key}' must be {typ.__name__}, got {type(output[key]).__name__}"
    return True

out = decide({"income": 60_000, "credit": 720})
print("provider output:", out)
print("contract honoured:", verify_contract(out, CONTRACT))

provider output: {'approved': True, 'score': 0.672}
contract honoured: True


☝ The contract test doesn't care *how* `decide` computes the answer — only that it always returns `{approved: bool, score: float}`. If a refactor renamed `approved` to `is_approved`, this test goes red *in the provider's own CI*, before the UI team ever sees a 500. That's the value of PACT-style contracts across a team: breaking changes are caught at the boundary that owns them.

## 4. Mutation testing — do the tests actually bite?

Passing tests can be *worthless* if they don't assert the right thing. Mutation testing checks the tests by **deliberately breaking the code**: flip an operator, and a good suite should now *fail* (the mutant is "killed"). A mutant that *survives* means you have a coverage blind spot.

In [4]:
def decide_mutant(app, threshold=0.5):
    return {"approved": score(app) > threshold, "score": round(score(app), 3)}   # >= mutated to >

def boundary_test(decide_fn):
    # applicant landing EXACTLY on the threshold: credit 999, income 0 -> score == 0.6
    app = {"income": 0, "credit": 999}
    return decide_fn(app, threshold=0.6)["approved"]

original = boundary_test(decide)
mutant   = boundary_test(decide_mutant)
print("original approves boundary case:", original)
print("mutant   approves boundary case:", mutant)
print("mutant KILLED (test caught it):", original != mutant)

original approves boundary case: True
mutant   approves boundary case: False
mutant KILLED (test caught it): True


☝ Changing `>=` to `>` flips the decision for an applicant *exactly on* the threshold — and our boundary test notices, so the mutant is **killed**. Had we only tested clearly-approved and clearly-rejected cases, this mutant would have *survived* and told us our suite never checks the boundary. Real tools (`mutmut`, `cosmic-ray`) generate hundreds of mutants automatically; the metric that matters is the **mutation kill rate**, not raw line coverage.

## 5. Quality gate — fail the build on regression

The final layer: run an eval suite (Day 15/24), compute a score, and **gate** on a threshold. In CI this is the line between "merge" and "blocked". Nightly, it's the alarm that the model or a prompt drifted.

In [5]:
EVAL_SET = [
    ({"income": 90_000, "credit": 810}, True),
    ({"income": 20_000, "credit": 400}, False),
    ({"income": 55_000, "credit": 650}, True),
    ({"income": 15_000, "credit": 300}, False),
]

def eval_gate(decide_fn, min_pass=0.75):
    correct = sum(decide_fn(app)["approved"] == expected for app, expected in EVAL_SET)
    rate = correct / len(EVAL_SET)
    status = "PASS ✅" if rate >= min_pass else "FAIL ❌ (blocks merge)"
    return rate, status

rate, status = eval_gate(decide)
print(f"eval accuracy: {rate:.0%}  ->  {status}")
rate_m, status_m = eval_gate(decide_mutant)
print(f"mutant accuracy: {rate_m:.0%}  ->  {status_m}")

eval accuracy: 100%  ->  PASS ✅
mutant accuracy: 100%  ->  PASS ✅


☝ The gate turns a fuzzy "seems fine" into a hard boolean the CI system can act on: below 75% accuracy, the merge is blocked — no human judgement required at 2am. Wire this into GitHub Actions on a nightly cron (with a tool like DeepEval driving richer metrics) and regressions get caught by the machine, not by a customer. A gate you don't enforce is just a dashboard.

## 6. Recap — four layers make agents safe to change

| Layer | Question it answers | Tool |
|---|---|---|
| Unit tests | does each node behave? | `pytest` |
| Contract tests | does output honour the consumer's shape? | PACT |
| Mutation tests | do the tests actually catch bugs? | `mutmut` / kill rate |
| Quality gate | did overall eval score regress? | eval suite + CI threshold |

Coverage tells you what code *ran*; mutation testing tells you what your tests actually *check*; the quality gate stops a regression from merging. Together they turn a non-deterministic agent into something you can refactor without fear. Day 29's Python lesson supplies the missing piece — **mocking the LLM** so these tests run fast and offline.